In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, r2_score
# Models
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
import xgboost as xgb
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.svm import LinearSVC

In [2]:
df = pd.read_parquet('../data/clean_data/clean_data.parquet')

In [3]:
df.drop(['age', 'partner', 'country', 'state', 'total_population', 'number_of_dependents', 'churn_score', 'churn_value', 'referred_a_friend'], axis=1, inplace=True)

In [4]:
df.to_parquet('../data/model_data/model_data.parquet')

In [5]:
x = df.drop('churn_label', axis=1)
y = df['churn_label']

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, stratify=y, random_state=42)

In [7]:
result = {}

In [8]:
y_train.value_counts()

churn_label
0    3864
1    1398
Name: count, dtype: int64

In [9]:
y_test.value_counts()

churn_label
0    1289
1     466
Name: count, dtype: int64

In [10]:
def decisiontree(x_train,x_test,y_train,y_test):
    decision__tree_params = {
        
        'max_depth':[None,3,5,10,20],
        'min_samples_split':[2,3,5,7,10],
        'min_samples_leaf':[1,2,3,4,5],
        'criterion':['gini', 'entropy'],
        
    }
    decision_model = DecisionTreeClassifier(random_state=42)
    grid_decision_model = GridSearchCV(estimator=decision_model, param_grid=decision__tree_params, cv=5, scoring='accuracy',verbose=1)
    grid_decision_model.fit(x_train, y_train)
    y_pred = grid_decision_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'decision_tree':{'accuracy':acc, 'best_params':grid_decision_model.best_params_}})
    return result['decision_tree']

In [11]:
# result = decisiontree(x_train, x_test, y_train, y_test)
# result

Fitting 5 folds for each of 250 candidates, totalling 1250 fits


{'accuracy': 0.9441595441595442,
 'best_params': {'criterion': 'entropy',
  'max_depth': 10,
  'min_samples_leaf': 5,
  'min_samples_split': 2}}

In [12]:
def randomforest(x_train,x_test,y_train,y_test):
    randomforest_params = {
        
        'n_estimators':[100, 200, 500, 1000],
        'max_depth':[None, 3, 5, 10, 20],
        'min_samples_split':[2, 3, 5, 7, 10],
        'min_samples_leaf':[1, 2, 3, 4, 5],
        'max_features':['sqrt', 'log2'],
        'bootstrap':[True, False]
    }
    randomforest_model = RandomForestClassifier(random_state=42)
    grid_randomforest_model = GridSearchCV(estimator=randomforest_model, param_grid=randomforest_params, cv=5, scoring='accuracy', verbose=1)
    grid_randomforest_model.fit(x_train, y_train)
    y_pred = grid_randomforest_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'randomforest':{'accuracy':acc, 'best_params':grid_randomforest_model.best_params_}})
    return result['randomforest']

In [13]:
def gradientboosting(x_train,x_test,y_train,y_test):
    gradient_params = {
        
        n_estimators:[1000],
        learning_rate:[0.001, 0.01, 0.5, 0.1],
        max_depth:[None, 3, 5, 7],
        min_samples_split:[2, 3, 5, 7, 10],
        min_samples_leaf:[1, 2, 3, 4, 5],
        subsample:[0.6, 0.8, 1]
        
    }
    gradient_model = GradientBoostingClassifier(random_state=42)
    grid_gradient_model = GridSearchCV(estimator=gradient_model, param_grid=gradient_params, cv=5, scoring='accuracy', verbose=1)
    grid_gradient_model.fit(x_train, y_train)
    y_pred = grid_gradient_model.predict(x_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'gradient_boosting':{'accuracy':acc, 'best_params':grid_gradient_model.best_params_}})
    return result['gradient_boosting']

In [17]:
def xgboost(x,y):
    xgb_params = {
    'n_estimators': [1000],               # Set high for early stopping
    'learning_rate': [0.01, 0.05, 0.1],  # Step size shrinkage
    'max_depth': [None, 3, 5, 7, 10],              # Tree complexity
    'subsample': [0.8, 1.0],             # Rows per tree
    'colsample_bytree': [0.8, 1.0],      # Columns per tree
    'grow_policy': ['depthwise', 'lossguide'],
    # Regularization
    'reg_alpha': [0, 0.1, 1],
    'reg_lambda': [0.1, 1, 10],
    'gamma': [0, 0.1, 1],
    # unbalanced
    'min_child_weight': [1, 5, 10],
    # 'eval_metric': ['logloss', 'aucpr']  # Handle class imbalance
    
}
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
    x_temp, x_val, y_temp, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    unbalanced_weight = round(y.value_counts().sort_values()[0]/y.value_counts().sort_values()[1])
    xgb_model = xgb.XGBClassifier(early_stopping_rounds=10, eval_metric='aucpr', scale_pos_weight=unbalanced_weight)
    grid_xgb_model = GridSearchCV(estimator=xgb_model, param_grid=xgb_params, cv=5, scoring='accuracy', n_jobs=-1)
    grid_xgb_model.fit(x_temp, y_temp, eval_set=[(x_val, y_val)])
    y_pred = grid_xgb_model.predit(y_test)
    acc = accuracy_score(y_test, y_pred)
    result.update({'xgboost':{'accuracy':acc, 'best_params':grid_xgb_model.best_params_}})
    return result['xgboost']

In [18]:
# random_sample = df.sample(frac=0.2)
# y.value_counts()

churn_label
0    1012
1     391
Name: count, dtype: int64

In [19]:
# x = random_sample.drop('churn_label',axis=1)
# y = random_sample['churn_label']
# xgboost(x,y)

[0]	validation_0-aucpr:0.94354
[0]	validation_0-aucpr:0.94513
[1]	validation_0-aucpr:0.94354
[1]	validation_0-aucpr:0.94500
[2]	validation_0-aucpr:0.94404
[2]	validation_0-aucpr:0.94500
[0]	validation_0-aucpr:0.96529
[3]	validation_0-aucpr:0.94417
[0]	validation_0-aucpr:0.96229
[3]	validation_0-aucpr:0.94500
[0]	validation_0-aucpr:0.96478
[4]	validation_0-aucpr:0.94417
[1]	validation_0-aucpr:0.96494
[1]	validation_0-aucpr:0.96498
[4]	validation_0-aucpr:0.94471
[1]	validation_0-aucpr:0.96447
[0]	validation_0-aucpr:0.95197
[5]	validation_0-aucpr:0.94417
[0]	validation_0-aucpr:0.93289
[2]	validation_0-aucpr:0.96498
[2]	validation_0-aucpr:0.96479
[2]	validation_0-aucpr:0.96447
[5]	validation_0-aucpr:0.94500
[0]	validation_0-aucpr:0.93816
[1]	validation_0-aucpr:0.95203
[6]	validation_0-aucpr:0.94544
[3]	validation_0-aucpr:0.96498
[3]	validation_0-aucpr:0.96479
[3]	validation_0-aucpr:0.96447
[1]	validation_0-aucpr:0.93816
[1]	validation_0-aucpr:0.93289
[6]	validation_0-aucpr:0.94642
[2]	vali

{'accuracy': 0.9373219373219374,
 'best_params': {'grow_policy': 'depthwise',
  'learning_rate': 0.1,
  'max_depth': 5,
  'min_child_weight': 1,
  'n_estimators': 1000,
  'reg_alpha': 1,
  'reg_lambda': 1}}

In [ ]:
def logistic(x_train,x_test,y_train,y_test):
    ...

In [ ]:
def knn(x_train,x_test,y_train,y_test):
    ...

In [ ]:
def gaussianNB(x_train,x_test,y_train,y_test):
    ...

In [ ]:
def linearsvc(x_train,x_test,y_train,y_test):
    ...